# Mục 3f — Thống kê sau tiền xử lý bằng Python

**Võ Minh Huy — 2311110117**

Đọc file after_Python.csv. Tính thống kê tương ứng với SQL trên bảng heart_after, chưa làm hai ma trận. Dùng Decimal để giữ các số thập phân từ CSV; chỉ làm tròn bảng hiển thị. Bảng SQL import bằng float có sai số nhỏ đã được chấp nhận, vì vậy không khẳng định hai bảng thống kê khớp tuyệt đối.

In [1]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP, localcontext
from collections import Counter
import json
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_after = thu_muc / "VoMinhHuy_Heart Failure Prediction_after_Python.csv"
du_lieu = pd.read_csv(file_after, dtype=str, keep_default_na=False)
thu_muc_kq = thu_muc / "ket_qua/19_thong_ke_3f"
thu_muc_kq.mkdir(parents=True, exist_ok=True)
quy_mo = pd.DataFrame({"SoHoSo":[len(du_lieu)], "SoCot":[du_lieu.shape[1]],
    "SoNhanHopLe":[du_lieu["DEATH_EVENT"].isin(["0", "1"]).sum()],
    "SoNhanChoXacMinh":[du_lieu["DEATH_EVENT"].eq("").sum()]})
quy_mo

,SoHoSo,SoCot,SoNhanHopLe,SoNhanChoXacMinh
0,299,18,298,1


## 1. Thống kê chín biến định lượng

Gồm bảy biến gốc và hai nhánh làm mịn platelets. Không tính mean/std cho ID, mã danh mục, mã nhóm tuổi hoặc mã Boolean. Phân vị nội suy tuyến tính ở vị trí (n−1)×p, cùng quy ước PERCENTILE_CONT. Std dùng mẫu số n−1. Nếu nhiều mode, chọn số nhỏ nhất và báo số mode đồng hạng.

In [2]:
cot_dinh_luong = ["age", "creatinine_phosphokinase", "ejection_fraction",
    "platelets", "serum_creatinine", "serum_sodium", "time",
    "platelets_mean_5_nhom", "platelets_dieu_chinh_10pct"]

def hai_so(value):
    return value.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

def phan_vi(x, p):
    vi_tri = Decimal(len(x) - 1) * Decimal(p)
    i = int(vi_tri)
    return x[i] + (vi_tri - i) * (x[min(i + 1, len(x) - 1)] - x[i])

dong_thong_ke = []
for cot in cot_dinh_luong:
    x = sorted(Decimal(v) for v in du_lieu[cot] if v.strip())
    if len(x) < 2:
        raise ValueError("Không đủ hai giá trị: " + cot)
    dem = Counter(x)
    tan_suat_max = max(dem.values())
    cac_mode = [v for v, n in dem.items() if n == tan_suat_max]
    with localcontext() as ctx:
        ctx.prec = 38
        mean = sum(x, Decimal(0)) / len(x)
        std = (sum((v - mean) ** 2 for v in x) / (len(x) - 1)).sqrt()
        dong_thong_ke.append({"TenCot":cot, "SoHopLe":len(x),
            "SoThieu":len(du_lieu)-len(x), "Mean":hai_so(mean),
            "ModeNhoNhat":hai_so(min(cac_mode)), "SoMode":len(cac_mode),
            "Median_Q2":hai_so(phan_vi(x,"0.50")), "StdMau":hai_so(std),
            "Min":hai_so(x[0]), "Q1":hai_so(phan_vi(x,"0.25")),
            "Q3":hai_so(phan_vi(x,"0.75")), "Max":hai_so(x[-1])})
dinh_luong = pd.DataFrame(dong_thong_ke)
pd.set_option("display.max_columns", None)
dinh_luong

,TenCot,SoHopLe,SoThieu,Mean,ModeNhoNhat,SoMode,Median_Q2,StdMau,Min,Q1,Q3,Max
0,age,299,0,60.74,60.00,1,60.00,11.25,40.00,53.00,68.00,95.00
1,creatinine_phosphokinase,299,0,552.71,582.00,1,249.00,958.09,23.00,120.00,582.00,7861.00
2,ejection_fraction,299,0,38.08,35.00,1,38.00,11.83,14.00,30.00,45.00,80.00
3,platelets,299,0,263868.72,263358.03,1,259763.43,94807.77,25100.00,217500.00,299000.00,850000.00
4,serum_creatinine,299,0,1.40,1.00,1,1.10,1.03,0.50,1.00,1.40,9.40
5,serum_sodium,299,0,136.45,137.00,1,137.00,4.30,113.00,134.00,139.00,148.00
6,time,299,0,130.26,187.00,2,115.00,77.61,4.00,73.00,203.00,285.00
7,platelets_mean_5_nhom,299,0,263868.72,261884.04,1,261884.04,86640.07,142198.04,261884.04,261884.04,796000.00
8,platelets_dieu_chinh_10pct,299,0,262862.74,263358.03,1,259763.43,90115.08,27610.00,217500.00,299000.00,765000.00


## 2. Tần suất và mode của các mã

Tỷ lệ tính trên toàn bộ 299 hồ sơ. Nhãn thiếu hiển thị riêng và không tham gia chọn mode. MaNhomTuoi chỉ biểu thị thứ tự; MaBoolean là mã tổ hợp, không phải mức độ bệnh.

In [3]:
cot_ma = ["anaemia", "diabetes", "high_blood_pressure", "sex", "smoking",
          "DEATH_EVENT", "MaNhomTuoi", "MaBoolean"]
dong_tan_suat = []
for cot in cot_ma:
    dem = Counter(None if v == "" else int(Decimal(v)) for v in du_lieu[cot])
    max_hop_le = max(n for v, n in dem.items() if v is not None)
    for ma in sorted(dem, key=lambda v: (v is not None, v if v is not None else -1)):
        so_luong = dem[ma]
        dong_tan_suat.append({"TenCot":cot, "Ma":"<NULL>" if ma is None else str(ma),
            "SoLuong":so_luong,
            "TyLePhanTram":hai_so(Decimal(100)*so_luong/len(du_lieu)),
            "GhiChu":"Mode" if ma is not None and so_luong == max_hop_le else ""})
tan_suat = pd.DataFrame(dong_tan_suat)
tan_suat.loc[tan_suat["TenCot"].ne("MaBoolean")].reset_index(drop=True)

,TenCot,Ma,SoLuong,TyLePhanTram,GhiChu
0,anaemia,0,170,56.86,Mode
1,anaemia,1,129,43.14,
2,diabetes,0,174,58.19,Mode
3,diabetes,1,125,41.81,
4,high_blood_pressure,0,194,64.88,Mode
5,high_blood_pressure,1,105,35.12,
6,sex,0,105,35.12,
7,sex,1,194,64.88,Mode
8,smoking,0,203,67.89,Mode
9,smoking,1,96,32.11,


In [4]:
tan_suat.loc[tan_suat["TenCot"].eq("MaBoolean")].reset_index(drop=True)

,TenCot,Ma,SoLuong,TyLePhanTram,GhiChu
0,MaBoolean,0,12,4.01,
1,MaBoolean,1,13,4.35,
2,MaBoolean,2,15,5.02,
3,MaBoolean,3,20,6.69,
4,MaBoolean,4,10,3.34,
5,MaBoolean,5,9,3.01,
6,MaBoolean,6,9,3.01,
7,MaBoolean,7,12,4.01,
8,MaBoolean,8,23,7.69,
9,MaBoolean,9,17,5.69,


## 3. Lưu kết quả và đối chiếu SQL

Đối chiếu tần suất theo cặp tên cột và mã, không theo thứ tự dòng. SQL sắp mã Boolean dạng chuỗi nên thứ tự có thể khác. Với thống kê định lượng, ghi rõ mọi chênh lệch ở độ chính xác hiển thị hai số lẻ; không sửa kết quả để ép khớp.

In [5]:
quy_mo.to_csv(thu_muc_kq / "quy_mo_python.csv", index=False, encoding="utf-8-sig")
dinh_luong.to_csv(thu_muc_kq / "dinh_luong_python.csv", index=False, encoding="utf-8-sig")
tan_suat.to_csv(thu_muc_kq / "tan_suat_python.csv", index=False, encoding="utf-8-sig")

sql = pd.read_csv(thu_muc_kq / "dinh_luong_sql.csv", dtype=str, keep_default_na=False).set_index("TenCot")
chenh_lech = []
for _, r in dinh_luong.iterrows():
    for chi_tieu in dinh_luong.columns.drop("TenCot"):
        py = Decimal(str(r[chi_tieu]))
        sq = Decimal(sql.at[r["TenCot"], chi_tieu])
        if py != sq:
            chenh_lech.append({"TenCot":r["TenCot"], "ChiTieu":chi_tieu,
                "SQL":sq, "Python":py, "ChenhLechTuyetDoi":abs(sq-py)})
bang_chenh = pd.DataFrame(chenh_lech, columns=["TenCot", "ChiTieu", "SQL", "Python", "ChenhLechTuyetDoi"])
bang_chenh.to_csv(thu_muc_kq / "chenh_lech_dinh_luong.csv", index=False, encoding="utf-8-sig")

sql_ma = pd.read_csv(thu_muc_kq / "tan_suat_sql.csv", dtype=str, keep_default_na=False)
def chuan_tan_suat(bang):
    return {(r.TenCot, r.Ma):(int(r.SoLuong), Decimal(str(r.TyLePhanTram)), r.GhiChu)
            for r in bang.itertuples(index=False)}
tan_suat_khop = chuan_tan_suat(sql_ma) == chuan_tan_suat(tan_suat)
bao_cao = {"so_ho_so":len(du_lieu), "so_bien_dinh_luong":len(dinh_luong),
    "so_dong_tan_suat":len(tan_suat), "tan_suat_khop_SQL":tan_suat_khop,
    "so_chi_tieu_dinh_luong_chenh":len(bang_chenh),
    "chenh_lech_hien_thi_lon_nhat":str(max((r["ChenhLechTuyetDoi"] for r in chenh_lech), default=Decimal(0))),
    "ghi_chu":"SQL dùng bảng import float; Python dùng số thập phân từ CSV. Không khẳng định thống kê khớp tuyệt đối."}
(thu_muc_kq / "doi_chieu_sql_python.json").write_text(json.dumps(bao_cao, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame([bao_cao])

,so_ho_so,so_bien_dinh_luong,so_dong_tan_suat,tan_suat_khop_SQL,so_chi_tieu_dinh_luong_chenh,chenh_lech_hien_thi_lon_nhat,ghi_chu
0,299,9,45,True,9,0.01,SQL dùng bảng import float; Python dùng số thậ...


In [6]:
bang_chenh

,TenCot,ChiTieu,SQL,Python,ChenhLechTuyetDoi
0,platelets,Median_Q2,259763.44,259763.43,0.01
1,platelets_mean_5_nhom,Mean,263868.73,263868.72,0.01
2,platelets_mean_5_nhom,ModeNhoNhat,261884.05,261884.04,0.01
3,platelets_mean_5_nhom,Median_Q2,261884.05,261884.04,0.01
4,platelets_mean_5_nhom,Min,142198.05,142198.04,0.01
5,platelets_mean_5_nhom,Q1,261884.05,261884.04,0.01
6,platelets_mean_5_nhom,Q3,261884.05,261884.04,0.01
7,platelets_dieu_chinh_10pct,Median_Q2,259763.44,259763.43,0.01
8,platelets_dieu_chinh_10pct,StdMau,90115.07,90115.08,0.01


**Phạm vi:** Đã tính thống kê mô tả và lưu bảng đối chiếu. Các phép này chỉ đọc dữ liệu; giữ nguyên hai file after. Một nhãn chưa xác minh vẫn thiếu. Hai ma trận cho ba hồ sơ chưa thực hiện trong notebook này.